# Noise playground for `generate_input_records_3.py`

This notebook reproduces the core per-TCE flow used in `generate_input_records_3.py` and lets you compare **clean** vs **noisy** inputs step by step.

Pipeline steps:
1. Load one TCE + light curve
2. Add optional Gaussian noise
3. Detrend
4. Compute scatter weights
5. Phase-fold + align raw time/weights
6. Build global/local views
7. Plot differences


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astronet.preprocess import preprocess

plt.rcParams["figure.figsize"] = (12, 4)
np.set_printoptions(suppress=True, precision=4)


In [ ]:
# --- Configure paths and target TCE ---
# Update these if needed for your environment.
TCE_CSV = Path("../../tce_info/tces-vetting-set.csv")
TESS_DATA_DIR = Path("/pdo/users/astronet/pipelines/Astronet/input_data")
FLUX_KEY = "SAP_FLUX"  # SAP_FLUX, SAP_FLUX_SML, SAP_FLUX_MID, SAP_FLUX_LAG

# Set to a specific Astro ID, or keep None to use the first row.
TARGET_ASTRO_ID = None

# Core params from generate_input_records_3.py
BKSPACE = 0.3
NOISE_STD = 0.0  # set > 0 to inject additive Gaussian noise in raw flux
RANDOM_SEED = 42


In [ ]:
df = pd.read_csv(TCE_CSV, low_memory=False)

if TARGET_ASTRO_ID is None:
    row = df.iloc[0]
else:
    match = df[df["Astro ID"] == TARGET_ASTRO_ID]
    if len(match) == 0:
        raise ValueError(f"Astro ID {TARGET_ASTRO_ID} not found in {TCE_CSV}")
    row = match.iloc[0]

astro_id = int(row["Astro ID"])
tic_id = int(row["TIC ID"])
period = float(row["Per"])
epoch = float(row["Epoc"])
duration = float(row["Dur"])

min_t = row["MinT"] if "MinT" in row and pd.notna(row["MinT"]) else -np.inf
max_t = row["MaxT"] if "MaxT" in row and pd.notna(row["MaxT"]) else np.inf
filename = row["File"]

raw_time, raw_flux = preprocess.read_and_process_light_curve(
    str(TESS_DATA_DIR),
    FLUX_KEY,
    filename,
    min_t,
    max_t,
)

print(f"Astro ID: {astro_id} | TIC ID: {tic_id}")
print(f"N points: {len(raw_time)} | period={period:.6f} d | epoch={epoch:.6f} | duration={duration:.6f} d")


In [ ]:
def run_v3_steps(
    tic_id,
    time,
    flux,
    period,
    epoch,
    duration,
    bkspace=0.3,
    noise_std=0.0,
    seed=42,
):
    """Run the main generate_input_records_3 preprocessing steps for one LC."""
    rng = np.random.default_rng(seed)
    flux_noisy = flux + rng.normal(0.0, noise_std, size=len(flux))

    detr_t, detr_f, tr_mask = preprocess.detrend_and_filter(
        tic_id, time, flux_noisy, period, epoch, duration, bkspace
    )

    weights_detr = preprocess.split_and_calculate_weights(detr_t, detr_f, gap_width=2)

    fold_t, fold_f, fold_num, fold_tr_mask = preprocess.phase_fold_and_sort_light_curve(
        detr_t, detr_f, tr_mask, period, epoch
    )

    raw_t_aligned, raw_f_aligned = preprocess.align_raw_time(detr_t, detr_f, period, epoch)
    weights_aligned = preprocess.align_scatter_weights(detr_t, period, epoch, weights_detr)

    g_view, g_std, g_mask, _, _ = preprocess.global_view(
        tic_id,
        fold_t,
        fold_f,
        period,
        all_30min=False,
        raw_time=raw_t_aligned,
        raw_flux=raw_f_aligned,
        scatter_weights=weights_aligned,
    )

    l_view, l_std, l_mask, l_scale, l_depth = preprocess.local_view(
        tic_id,
        fold_t,
        fold_f,
        period,
        duration,
        all_30min=False,
        raw_time=raw_t_aligned,
        raw_flux=raw_f_aligned,
        scatter_weights=weights_aligned,
    )

    (s_view, s_std, s_mask, _, _), s_t0 = preprocess.secondary_view(
        tic_id,
        fold_t,
        fold_f,
        period,
        duration,
        scale=l_scale,
        depth=l_depth,
        all_30min=False,
        raw_time=raw_t_aligned,
        raw_flux=raw_f_aligned,
    )

    return {
        "flux_noisy": flux_noisy,
        "detr_t": detr_t,
        "detr_f": detr_f,
        "tr_mask": tr_mask,
        "weights_detr": weights_detr,
        "fold_t": fold_t,
        "fold_f": fold_f,
        "fold_num": fold_num,
        "fold_tr_mask": fold_tr_mask,
        "raw_t_aligned": raw_t_aligned,
        "raw_f_aligned": raw_f_aligned,
        "weights_aligned": weights_aligned,
        "global_view": g_view,
        "global_std": g_std,
        "global_mask": g_mask,
        "local_view": l_view,
        "local_std": l_std,
        "local_mask": l_mask,
        "secondary_view": s_view,
        "secondary_std": s_std,
        "secondary_mask": s_mask,
        "secondary_t0": s_t0,
    }


In [ ]:
# Baseline (no noise) vs noisy run
clean = run_v3_steps(tic_id, raw_time, raw_flux, period, epoch, duration, bkspace=BKSPACE, noise_std=0.0, seed=RANDOM_SEED)
noisy = run_v3_steps(tic_id, raw_time, raw_flux, period, epoch, duration, bkspace=BKSPACE, noise_std=NOISE_STD, seed=RANDOM_SEED)

print(f"NOISE_STD={NOISE_STD}")
print(f"Detrended points (clean/noisy): {len(clean['detr_t'])}/{len(noisy['detr_t'])}")
print(f"Global view mean abs diff: {np.mean(np.abs(clean['global_view'] - noisy['global_view'])):.6e}")
print(f"Local view mean abs diff:  {np.mean(np.abs(clean['local_view'] - noisy['local_view'])):.6e}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(raw_time, raw_flux, ".", ms=1, alpha=0.6, label="raw clean")
axes[0].plot(raw_time, noisy["flux_noisy"], ".", ms=1, alpha=0.6, label="raw + noise")
axes[0].set_title("Raw light curve")
axes[0].set_xlabel("time")
axes[0].legend()

axes[1].plot(clean["detr_t"], clean["detr_f"], ".", ms=1, alpha=0.7, label="detrended clean")
axes[1].plot(noisy["detr_t"], noisy["detr_f"], ".", ms=1, alpha=0.7, label="detrended noisy")
axes[1].set_title("Detrended")
axes[1].set_xlabel("time")
axes[1].legend()

axes[2].plot(clean["detr_t"], clean["weights_detr"], ".", ms=2, alpha=0.8, label="weights clean")
axes[2].plot(noisy["detr_t"], noisy["weights_detr"], ".", ms=2, alpha=0.8, label="weights noisy")
axes[2].set_title("Scatter weights on detrended curve")
axes[2].set_xlabel("time")
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(clean["global_view"], lw=2, label="global clean")
axes[0].plot(noisy["global_view"], lw=1.5, label="global noisy")
axes[0].set_title("Global view")
axes[0].legend()

axes[1].plot(clean["local_view"], lw=2, label="local clean")
axes[1].plot(noisy["local_view"], lw=1.5, label="local noisy")
axes[1].set_title("Local view")
axes[1].legend()

axes[2].plot(clean["secondary_view"], lw=2, label="secondary clean")
axes[2].plot(noisy["secondary_view"], lw=1.5, label="secondary noisy")
axes[2].set_title("Secondary view")
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Quick manual playground:
# 1) Change NOISE_STD in config cell (e.g. 0.0001, 0.001, 0.005)
# 2) Re-run baseline/noisy + plots
# 3) If you want to inspect each array:
#    clean.keys(), clean['weights_aligned'][:10], etc.
pass
